In [2]:

from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
import requests

load_dotenv()


True

In [3]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke("top news in pakistan today")
results

/tmp/ipykernel_37544/3609738333.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


'12 hours ago - Islamabad is using a domestic insurgency to justify its broader pursuit of strategic dominance. Mountaineer Nirmal Purja among those reported missing after avalanche hit Broad Peak, Pakistan alpine club says. Monsoon death toll rises to 126 as electrocution tops causes of fatalities Updated 16 hours ago ... Polling ended in eight Muzaffarabad constituencies and 12 refugee seats as PPP alleged rigging by the PML-N ... Moghadam says ties with Pakistan ... 1 day ago - Karachi shopping centre fire: 11-year-old named accused in chargesheet over blaze that killed 72 · PoK erupts as police open fire on protesters amid massive anti-Pakistan rallies, one killed - key points ... Coronavirus NewsJoe BidenUS NewsPakistan NewsDonald TrumpChina NewsSouth Asia NewsWorld NewsBoris Johnson 7 hours ago - Language & regionEnglish (Pakistan) ... weather · 97°FGoogle Weather · radio.gov.pk · More · Govt revises prices of petroleum products for next three days ·... 2 days ago - Against the o

In [8]:
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain_classic import hub
from langchain_core.prompts import PromptTemplate


In [9]:
# Step 2. Custom ReAct prompt to instruct llm Reasoning + Acting 
REACT_TEMPLATE = """Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format exactly:

Question: the input question you must answer
Thought: think about what to do next
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result returned by the tool
... (this Thought/Action/Action Input/Observation sequence can repeat as many times as needed)
Thought: I now know the final answer
Final Answer: the final answer to the original question

Rules:
- Always think before acting.
- Only use the available tools.
- Never invent tool outputs.
- If you already have enough information, do not call another tool.
- The Final Answer should be concise and directly answer the user's question.

Begin!

Question: {input}
Thought: {agent_scratchpad}
"""

prompt = PromptTemplate.from_template(REACT_TEMPLATE)



In [10]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", temperature=0.7 
)

In [ ]:
# Step 3: create react agent
# agent decided when or what tool to use
agent = create_react_agent(
    llm=llm,
    tools=[search_tool],
    prompt=prompt
)

In [13]:
# Step 4: Wrap it with Agent Executor
# Agent Executor listens to agent, agent tells agent executor to what tool to execute
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool],
    verbose=True
)

In [14]:
# Step 5: execute
response = agent_executor.invoke({"input": "3 ways to reach skardu from lahore punjab"})
response



> Entering new AgentExecutor chain...
Action: duckduckgo_search
Action Input: how to reach skardu from lahoreEverything about Lahore to Skardu flights — airlines, ticket prices, schedules, cancellation risks, and what to expect after landing at Skardu Airport Whether you're planning an adventurous road trip or a quick flight, knowing the best way to reach Skardu can make your journey smooth and enjoyable. This complete travel guide explains every transportation option from Pakistan's major cities, estimated travel time, costs, and practical travel tips. Traveling from Lahore to Skardu opens the gateway to one of Pakistan's most breathtaking regions, Gilgit-Baltistan. With its stunning mountains, lakes, and unique culture, Skardu has become a favorite destination for adventure seekers and leisure travelers alike. For beginners, flying is the most practical option, as road travel is long and challenging. This guide will cover everything you ... Lahore to Skardu Flights Travelers from L

{'input': '3 ways to reach skardu from lahore punjab',
 'output': '1. Direct seasonal flights from Lahore to Skardu. 2. Connecting flights from Lahore to Skardu via Islamabad. 3. Road travel from Lahore to Skardu.'}